# Fine-tuning real do YOLOv8n sobre Endoscapes-BBox201 (GPU, Colab)

Este notebook **não reimplementa** a lógica de treino de F1 — ele importa e chama
`backend/pipelines/video/object_finetune.py` (já testado, 362 testes verdes, Verifier PASS)
rodando na GPU do Colab em vez da CPU local (AD-042). A inferência do sistema continua
100% CPU (AD-012); só esta etapa offline de treino usa GPU.

**Antes de abrir este notebook**: rode localmente
`PYTHONPATH=backend .venv/bin/python training/prepare_dataset_subset.py`, zipe
`training/staging/` e suba o zip para o seu Google Drive (ver `training/README.md`).
Isso evita subir o dataset bruto de ~6 GB — só as 1933 imagens realmente anotadas (~207 MB).

**Runtime**: Ambiente de execução -> Alterar tipo de ambiente de execução -> GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ajuste para o caminho onde você subiu o zip do passo anterior.
DRIVE_STAGING_ZIP = '/content/drive/MyDrive/endoscapes_staging.zip'
# Tudo que este notebook produz (pesos, métricas) fica no Drive -- a sessão do
# Colab é efêmera e cai por inatividade (requisito de persistência, AD-042).
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/endoscapes_training_output'

In [ ]:
# Código de F1 (sem reimplementação) -- clona só para importar
# `pipelines.video.object_finetune`/`object_loader`/`object_detector`/`object_evaluate`.
!git clone --depth 1 https://github.com/AnaPRodrigues/8iadt-tc-fase4-multi-monitoring.git /content/repo
!pip install -q -r /content/repo/training/requirements-training.txt

import sys
sys.path.insert(0, '/content/repo/backend')

In [ ]:
import shutil
from pathlib import Path

staging_dir = Path('/content/endoscapes_staging')
shutil.unpack_archive(DRIVE_STAGING_ZIP, staging_dir)

# Splits oficiais do Endoscapes (requisito 4: nunca embaralhar frames entre
# treino e teste -- frames vizinhos de um mesmo vídeo são quase idênticos).
train_coco = staging_dir / 'train' / 'annotation_coco.json'
train_images = staging_dir / 'train'
test_coco = staging_dir / 'test' / 'annotation_coco.json'
test_images = staging_dir / 'test'

assert train_coco.is_file(), f'{train_coco} ausente -- confira o zip subido ao Drive'
assert test_coco.is_file(), f'{test_coco} ausente -- confira o zip subido ao Drive'

## Treino

`object_finetune.finetune()` (código real de F1) converte o COCO para YOLO e treina.
**Nota de metodologia (AD-042)**: internamente `finetune()` usa o próprio split de
treino também como validação de treino (early-stopping fraco) -- isso é comportamento
já existente de F1, não alterado aqui. A métrica que de fato reportamos no relatório
vem da célula de avaliação abaixo, rodada contra o split `test/` oficial, nunca visto
durante o treino -- é isso que garante a métrica honesta exigida pelo requisito 4.

In [ ]:
from pipelines.video.object_finetune import finetune

SEED = 42
EPOCHS = 50
IMGSZ = 640

import torch
torch.manual_seed(SEED)

best_weights = finetune(
    coco_json=train_coco,
    images_dir=train_images,
    output_dir=Path(DRIVE_OUTPUT_DIR),  # grava direto no Drive -- sobrevive a queda de sessão
    epochs=EPOCHS,
    imgsz=IMGSZ,
)
print('best.pt em:', best_weights)

In [ ]:
# results.csv do ultralytics fica ao lado de weights/ dentro do run -- copia para
# a raiz do output (mais fácil de achar depois).
results_csv = best_weights.parent.parent / 'results.csv'
if results_csv.is_file():
    shutil.copy2(results_csv, Path(DRIVE_OUTPUT_DIR) / 'results.csv')
    print('results.csv salvo em', Path(DRIVE_OUTPUT_DIR) / 'results.csv')

## Avaliação honesta contra o split `test/` (nunca visto no treino)

Reusa `object_loader.load_annotated_frames` (rótulo real) + `object_detector.YoloDetector`
(inferência com o `best.pt` recém-treinado) + `object_evaluate.evaluate` (precision/recall/F1
por classe via casamento de IoU >= 0.5) -- os três já existem em F1, nenhum reimplementado.

In [ ]:
import json
from dataclasses import asdict

from pipelines.video.object_loader import load_annotated_frames
from pipelines.video.object_detector import YoloDetector
from pipelines.video.object_evaluate import evaluate

annotated_frames = load_annotated_frames(test_coco, test_images)

detector = YoloDetector(best_weights)
detections_by_frame = {
    frame.image_path: detector.detect(frame.image_path) for frame in annotated_frames
}

reports = evaluate(annotated_frames, detections_by_frame)

metrics_path = Path(DRIVE_OUTPUT_DIR) / 'metrics_test.json'
metrics_path.write_text(
    json.dumps({k: asdict(v) for k, v in reports.items()}, indent=2, ensure_ascii=False),
    encoding='utf-8',
)

for classe, relatorio in reports.items():
    print(f"{classe}: precision={relatorio.precision} recall={relatorio.recall} f1={relatorio.f1} support={relatorio.support}")
print('\nmetrics_test.json salvo em', metrics_path)

## Próximos passos (fora do Colab)

1. Baixe `best.pt`, `results.csv` e `metrics_test.json` de `DRIVE_OUTPUT_DIR` para
   `models/` local.
2. Preencha `models/README.md` com proveniência, hiperparâmetros, seed e as métricas
   de `metrics_test.json`.
3. Publique `best.pt` como asset de um GitHub Release do repositório.
4. `make models-fetch` baixa esse Release para quem só quer rodar a demo sem retreinar.